# Week 6 — Block 2: Guided Demo (Graph & Network Data)

**DATS 6401 · Visualization of Complex Data**

This is the live-coding notebook for the ~30-minute guided demonstration. Students follow along; the instructor drives and narrates.

**Agenda** (mirrors the Block 1 deck):
1. Build a graph from an edge list (~10 min)
2. Compute today's measures — degree, betweenness, communities — one line each (~5 min)
3. Draw it: spring layout, degree → size, community → color (~10 min)
4. The adjacency-matrix view in two lines (~5 min)


## Part 1 — From edge list to graph (~10 min)

Real network data almost always arrives as an **edge list**: a table with a `source` and a `target` column. Watch how little it takes to turn that into a graph object.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# A tiny edge list, exactly the shape your data will have
edges = pd.DataFrame({
    "source": ["Ana", "Ana", "Ben", "Cal", "Cal", "Dee", "Eve"],
    "target": ["Ben", "Cal", "Cal", "Dee", "Eve", "Eve", "Fay"],
})
edges

In [ ]:
# One line: edge list -> graph
G_small = nx.from_pandas_edgelist(edges, "source", "target")

print("Nodes:", G_small.number_of_nodes())
print("Edges:", G_small.number_of_edges())
print("Degrees:", dict(G_small.degree()))

**Narrate:** point out that `degree` already tells a story — Cal touches four edges, Fay only one. Cal is a hub in miniature.

Now switch to the richer running example from the deck: **Zachary's Karate Club** — 34 members of a university karate club, edges = friendships, famous because the club later split into two factions.

In [ ]:
G = nx.karate_club_graph()
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())

# Each node carries the faction it ended up in after the split — we will NOT use
# this for community detection; it's our ground truth to check against.
print("Node 0 (the instructor):", G.nodes[0])
print("Node 33 (the president):", G.nodes[33])

## Part 2 — Today's measures, one line each (~5 min)

The Block 1 deck defined degree, betweenness, and community detection. Here is each one as a single line of networkx.

In [ ]:
# Degree: count the connections
deg = dict(G.degree())

# Betweenness: fraction of shortest paths passing through each node (Brandes' algorithm)
btw = nx.betweenness_centrality(G)

# Communities: greedy modularity maximization (Clauset-Newman-Moore)
from networkx.algorithms.community import greedy_modularity_communities, modularity
comms = list(greedy_modularity_communities(G))

print("Detected", len(comms), "communities")
print("Modularity Q =", round(modularity(G, comms), 2))
print("Top 3 by degree:     ", sorted(deg, key=deg.get, reverse=True)[:3])
print("Top 3 by betweenness:", sorted(btw, key=btw.get, reverse=True)[:3])

**Narrate:** nodes 0 and 33 — the instructor and the president — top both lists. The structure alone finds the protagonists of the split. Also note Q = 0.41, comfortably above the ~0.3 "meaningful structure" threshold from the deck.

## Part 3 — Draw it with encodings (~10 min)

The layout consumes x/y, so we encode **computed structure** on the remaining channels:
- node **size** ← degree
- node **color** ← detected community

In [ ]:
# Map each node to its community id
cmap = {}
for cid, community in enumerate(comms):
    for n in community:
        cmap[n] = cid

# Force-directed layout. ALWAYS set a seed: layouts are algorithms, not facts.
pos = nx.spring_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(9, 5.5))
nx.draw_networkx(
    G, pos, ax=ax, with_labels=False,
    node_size=[deg[n] * 40 for n in G.nodes()],
    node_color=[cmap[n] for n in G.nodes()],
    cmap=plt.cm.Set2,
    edge_color="#dddddd",
)
ax.set_title("Karate club — size = degree, color = detected community")
ax.axis("off")
plt.show()

**Demonstrate the layout caution from the deck** — re-run with a different seed and watch the picture change while the data stays identical:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, seed in zip(axes, [42, 7]):
    p = nx.spring_layout(G, seed=seed)
    nx.draw_networkx(G, p, ax=ax, with_labels=False,
                     node_size=[deg[n] * 30 for n in G.nodes()],
                     node_color=[cmap[n] for n in G.nodes()],
                     cmap=plt.cm.Set2, edge_color="#dddddd")
    ax.set_title(f"spring_layout(seed={seed})")
    ax.axis("off")
plt.show()

**Ask the class:** which of these is the "correct" drawing? (Neither — both are valid interpretations of the same 78 edges. This is why we caption layouts and fix seeds.)

## Part 4 — The matrix view in two lines (~5 min)

When node–link starts to clutter, the **adjacency matrix** is the honest alternative.

In [ ]:
A = nx.to_numpy_array(G)
plt.imshow(A, cmap="Blues")
plt.title("Adjacency matrix — every blue cell is one of the 78 edges")
plt.xlabel("node j"); plt.ylabel("node i")
plt.colorbar(label="edge")
plt.show()

## Wrap-up → Block 3

You've seen the full pipeline: **edge list → graph → measures → encoded drawing → matrix alternative.**

After the break, it's your turn — same pipeline on the **Les Misérables** character network, ending with an interactive PyVis network inside a **Streamlit** app. Open `week06_block3_handson.ipynb`.